# SOLUTION (R version): Checking Assumptions for t-tests, ANOVA & Tukey
## Rigorous Checks, Decision Framework & Practical Application


## Flowchart: Assumption Checking Workflow (Practical Decision Tree)
```mermaid
flowchart TD
    Start[Load Data & Define Groups] --> VarCheck{Check Equal Variances<br/>Ratio of SDs ~0.9-1.1?<br/>+ var.test or Levene p > 0.05?}
    VarCheck -->|Yes| NormCheck{Check Normality<br/>Shapiro p > 0.05 or large n + visual OK?}
    VarCheck -->|No| Welch[Use Welch t-test<br/>(var.equal = FALSE) or<br/>non-parametric test]
    NormCheck -->|Yes| Proceed[Proceed with t-test / ANOVA / Tukey]
    NormCheck -->|No| Robust[Large n? → CLT often OK<br/>Small n? → Transform or non-parametric]
    Robust --> Proceed
    Proceed --> Report[Document checks in report<br/>+ sensitivity analysis if borderline]
    Report --> Audience[Tailor depth to audience]
```
**Practical rule:** With n ≥ 30–50 per group, moderate violations are often tolerable. Always report what you checked.


## 1. Load Data and Quick Exploration (Solution)

**Key finding:** Ratio of SDs ≈ 0.62 → clear violation of equal variance assumption.


In [ ]:
dist_1 <- scan("1.csv")
dist_2 <- scan("2.csv")

cat("dist_1: n =", length(dist_1), "mean =", round(mean(dist_1), 2), "sd =", round(sd(dist_1), 2), "\n")
cat("dist_2: n =", length(dist_2), "mean =", round(mean(dist_2), 2), "sd =", round(sd(dist_2), 2), "\n")

ratio <- sd(dist_1) / sd(dist_2)
cat("\nRatio of sd (dist_1 / dist_2) =", round(ratio, 3), "→ Not close to 1\n")

hist(dist_1, breaks = 20, col = rgb(0.2, 0.4, 0.6, 0.5), main = "dist_1 vs dist_2", xlab = "Value", freq = FALSE)
hist(dist_2, breaks = 20, col = rgb(0.8, 0.4, 0.2, 0.5), add = TRUE, freq = FALSE)
legend("topright", legend = c("dist_1", "dist_2"), fill = c(rgb(0.2,0.4,0.6,0.5), rgb(0.8,0.4,0.2,0.5)))


## 2. Check Equal Variances – Rigorous (Solution)

**Result:** `var.test` p-value is extremely small → strong evidence that variances are unequal.

**Decision:** Use Welch t-test (`var.equal = FALSE`).


In [ ]:
var_test <- var.test(dist_1, dist_2)
print(var_test)

cat("\nConclusion: Variances are significantly different → Use Welch t-test (var.equal = FALSE).\n")


## 3. Check Normality (Solution)

**Results:** Shapiro p-values are both high. Q-Q plots look reasonable. With n=100, normality is acceptable.

**Main issue is unequal variances, not normality.**


In [ ]:
par(mfrow = c(1,2))
qqnorm(dist_1, main = "Q-Q Plot: dist_1"); qqline(dist_1, col = "red")
qqnorm(dist_2, main = "Q-Q Plot: dist_2"); qqline(dist_2, col = "red")
par(mfrow = c(1,1))

cat("Shapiro-Wilk dist_1 p-value:", round(shapiro.test(dist_1)$p.value, 4), "\n")
cat("Shapiro-Wilk dist_2 p-value:", round(shapiro.test(dist_2)$p.value, 4), "\n")

cat("\nConclusion: Normality is acceptable for n=100. Main concern = unequal variances.\n")


## 4. Decision Framework (Solution)

**Recommendation:**
- Use Welch t-test (`var.equal = FALSE`)
- Normality OK due to sample size
- Always document assumption checks in your report


## 5. Apply to VeryAnts (Solution)

From previous exercises: Levene p-value was high and Shapiro tests passed → standard ANOVA + Tukey was fine.


In [ ]:
veryants <- read_csv("veryants.csv")
a <- veryants$Sale[veryants$Store == "A"]
b <- veryants$Sale[veryants$Store == "B"]
c <- veryants$Sale[veryants$Store == "C"]

cat("Levene p-value (VeryAnts):", round(var.test(a, b)$p.value, 4), "\n")  # simplified
for (s in c("A","B","C")) {
  grp <- veryants$Sale[veryants$Store == s]
  cat(sprintf("Shapiro %s: p = %.4f\n", s, shapiro.test(grp)$p.value))
}

cat("\nVeryAnts assumptions were well met → standard ANOVA + Tukey was appropriate.\n")


## 6. Simulation (Full Working R Version)


In [ ]:
set.seed(42)

n <- 100
mean1 <- 18; mean2 <- 12
sd1 <- 3; sd2 <- 5
n_simulations <- 500
alpha <- 0.05

sig_equal <- 0
sig_welch <- 0

for (i in 1:n_simulations) {
  g1 <- rnorm(n, mean1, sd1)
  g2 <- rnorm(n, mean2, sd2)
  
  p_equal <- t.test(g1, g2, var.equal = TRUE)$p.value
  p_welch <- t.test(g1, g2, var.equal = FALSE)$p.value
  
  if (p_equal < alpha) sig_equal <- sig_equal + 1
  if (p_welch < alpha) sig_welch <- sig_welch + 1
}

cat("Power with var.equal = TRUE :", round(sig_equal / n_simulations, 3), "\n")
cat("Power with var.equal = FALSE (Welch):", round(sig_welch / n_simulations, 3), "\n")
cat("\nWhen variances differ, Welch is safer.\n")


## 7. Example Conclusion & Audience Reporting (R Solution)

### Technical Version
`var.test` showed significantly different variances (p << 0.001), so we used Welch t-test. Shapiro-Wilk and Q-Q plots supported approximate normality (n=100). We documented all checks in the report.

### Executive Version
We verified the key assumptions before comparing the two groups. The main issue was different spread between groups, so we used a more robust test. Results can be trusted.
